# Simulation Data Scaling Analysis

This notebook analyzes the results of simulation experiments with different data configurations.
It reads from `best_metrics.json` files generated during training.


In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Configuration
CHECKPOINT_ROOT = Path("../../logs/pose_estimation/simulation/checkpoints")

def load_simulation_results(checkpoint_root: Path) -> pd.DataFrame:
    """
    Load all best_metrics.json files from simulation checkpoints.
    
    Parses experiment names like:
    - sim_k4_q85_base -> k=4, q=0.85
    - sim_k10_q9_base -> k=10, q=0.9
    - sim_k10_q85_60pct_base -> k=10, q=0.85, data_pct=0.6
    """
    records = []
    
    # Patterns for experiment names
    # Pattern 1: with pct (e.g., sim_k4_q950105_100pct_base-...)
    # Dataset stops before _\d+pct_
    pattern_with_pct = re.compile(
        r"^PointTransformer-sim_(?P<dataset>[^_]+(?:_(?!\d+pct)[^_]+)*)_(?P<pct>\d+)pct_(?P<scale>\w+)-"
    )
    # Pattern 2: without pct (e.g., sim_k4_q950105_base-...)
    # Dataset stops before _scale-
    pattern_no_pct = re.compile(
        r"^PointTransformer-sim_(?P<dataset>[^_]+(?:_(?!base|large|small|tiny|micro|nano)[^_]+)*)_(?P<scale>base|large|small|tiny|micro|nano)-"
    )
    
    for folder in checkpoint_root.iterdir():
        if not folder.is_dir():
            continue
        
        metrics_file = folder / "best_metrics.json"
        if not metrics_file.exists():
            continue
        
        # Parse folder name - try pattern with pct first
        match = pattern_with_pct.match(folder.name)
        if match:
            dataset = match.group("dataset")
            pct_str = match.group("pct")
            scale = match.group("scale")
        else:
            # Try pattern without pct
            match = pattern_no_pct.match(folder.name)
            if match:
                dataset = match.group("dataset")
                pct_str = None
                scale = match.group("scale")
            else:
                print(f"Skipping (no match): {folder.name}")
                continue
        
        # Parse k and q from dataset name
        ds_match = re.match(r"k(\d+)_q(\d+)", dataset)
        if ds_match:
            k_value = int(ds_match.group(1))
            q_raw = int(ds_match.group(2))
            # Convert q: q85 -> 0.85, q9 -> 0.9, q95 -> 0.95
            q_value = q_raw / 100.0 if q_raw >= 10 else q_raw / 10.0
        else:
            k_value = None
            q_value = None
        
        # Data percentage (default 100%)
        data_pct = int(pct_str) / 100.0 if pct_str else 1.0
        
        # Load metrics
        try:
            with open(metrics_file) as f:
                metrics = json.load(f)
            
            record = {
                "folder": folder.name,
                "dataset": dataset,
                "k": k_value,
                "q": q_value,
                "data_pct": data_pct,
                "scale": scale,
                "best_epoch": metrics.get("best_epoch"),
                "train_mpjpe": metrics.get("train", {}).get("mpjpe"),
                "train_pmpjpe": metrics.get("train", {}).get("p_mpjpe"),
                "test_mpjpe": metrics.get("test", {}).get("mpjpe"),
                "test_pmpjpe": metrics.get("test", {}).get("p_mpjpe"),
            }
            records.append(record)
            print(f"Loaded: {dataset} (k={k_value}, q={q_value}, pct={data_pct:.0%}) -> test_mpjpe={record['test_mpjpe']:.2f}")
            
        except Exception as e:
            print(f"Error loading {metrics_file}: {e}")
    
    return pd.DataFrame(records)

# Load all results
df = load_simulation_results(CHECKPOINT_ROOT)
print(f"\nLoaded {len(df)} experiments")
df.sort_values(["dataset", "data_pct"], inplace=True)
df


In [ ]:
# Summary statistics by dataset
print("=" * 80)
print(f"{'Dataset':<15} {'K':<5} {'Q':<6} {'Data%':<8} {'Test MPJPE':<12} {'Test P-MPJPE':<12}")
print("=" * 80)

for _, row in df.iterrows():
    print(f"{row['dataset']:<15} {row['k']:<5} {row['q']:<6.2f} {row['data_pct']*100:<8.0f} {row['test_mpjpe']:<12.2f} {row['test_pmpjpe']:<12.2f}")

print("=" * 80)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from cycler import cycler

# ----------------- Matplotlib Global Style Configuration -----------------
plt.rcdefaults()

plt.rcParams["figure.figsize"] = [8.0, 5.0]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300

plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"

plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['lines.markersize'] = 8
plt.rcParams['lines.markeredgewidth'] = 2.0

plt.rcParams["font.size"] = 18
plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams["legend.fontsize"] = "medium"
plt.rcParams["legend.facecolor"] = "white"
plt.rcParams["legend.edgecolor"] = "white"
plt.rcParams["legend.framealpha"] = 0.9
plt.rcParams['legend.frameon'] = False
plt.rcParams['legend.handlelength'] = 1.5
plt.rcParams['legend.handletextpad'] = 0.5
plt.rcParams['legend.columnspacing'] = 0.8
plt.rcParams['legend.labelspacing'] = 0.3

# Color palette
palette = ['#1e90ff', '#ffbb00', '#ff5080', '#a7426d', '#ff3c10', "#282828", "#00aa55"]
markers = ['o', 's', '^', 'D', 'v', 'p', 'h']
plt.rcParams["axes.prop_cycle"] = cycler(color=palette) + cycler(marker=markers)


In [ ]:
# Plot: Test MPJPE by Dataset (grouped bar chart)
fig, ax = plt.subplots(figsize=(10, 5))

# Filter to 100% data experiments only
df_full = df[df['data_pct'] == 1.0].copy()

if not df_full.empty:
    # Sort by test_mpjpe for better visualization
    df_full = df_full.sort_values('test_mpjpe')
    
    x = np.arange(len(df_full))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, df_full['test_mpjpe'], width, label='Test MPJPE', color='#1e90ff')
    bars2 = ax.bar(x + width/2, df_full['test_pmpjpe'], width, label='Test P-MPJPE', color='#ff5080')
    
    ax.set_ylabel('Error (mm)', fontsize=14)
    ax.set_xlabel('Dataset Configuration', fontsize=14)
    ax.set_title('Simulation Results by Dataset (100% Data)', fontsize=16)
    ax.set_xticks(x)
    ax.set_xticklabels(df_full['dataset'], rotation=45, ha='right')
    ax.legend(loc='upper right')
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('simulation_results_by_dataset.pdf', bbox_inches='tight', dpi=300)
plt.show()
print("Saved: simulation_results_by_dataset.pdf")


In [ ]:
# Plot: MPJPE vs Q value (grouped by K)
fig, ax = plt.subplots(figsize=(8, 5))

# Filter to 100% data experiments only
df_full = df[(df['data_pct'] == 1.0) & (df['k'].notna())].copy()

if not df_full.empty:
    # Group by K value
    for k_val in sorted(df_full['k'].unique()):
        df_k = df_full[df_full['k'] == k_val].sort_values('q')
        ax.plot(df_k['q'], df_k['test_mpjpe'], 
                label=f'k={int(k_val)}', linewidth=2.5, markersize=10)
    
    ax.set_xlabel('Quality Factor (q)', fontsize=14)
    ax.set_ylabel('Test MPJPE (mm)', fontsize=14)
    ax.set_title('MPJPE vs Quality Factor by K', fontsize=16)
    ax.legend(loc='upper right', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Set x-axis limits
    ax.set_xlim(0.8, 1.0)

plt.tight_layout()
plt.savefig('mpjpe_vs_quality.pdf', bbox_inches='tight', dpi=300)
plt.show()
print("Saved: mpjpe_vs_quality.pdf")


In [ ]:
# Data Scaling Analysis (if experiments with different data_pct exist)
# This cell is useful when running train_simulation_batch_different_size.py

def fit_power_law(x_data, y_data):
    """Fit y = a * x^b using linear regression on log-transformed data.
    Returns (a, b, r) where r is the correlation coefficient."""
    x_data = np.asarray(x_data)
    y_data = np.asarray(y_data)
    positive_indices = (x_data > 0) & (y_data > 0)
    if not np.any(positive_indices):
        return None, None, None
    x_pos = x_data[positive_indices]
    y_pos = y_data[positive_indices]
    log_x = np.log(x_pos)
    log_y = np.log(y_pos)
    b, log_a = np.polyfit(log_x, log_y, 1)
    a = np.exp(log_a)
    # Calculate correlation coefficient
    r = np.corrcoef(log_x, log_y)[0, 1]
    return a, b, r

# Check if we have data scaling experiments
df_scaling = df[df['data_pct'] < 1.0]

if not df_scaling.empty:
    print(f"Found {len(df_scaling)} data scaling experiments")
    
    # Assume total data points per dataset (can be adjusted)
    TOTAL_DATA_POINTS = 40000
    
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Label mapping for legend
    label_map = {'k4_q950105': 'WiCompass', 'mmbody0105': 'mmBody trace'}
    datasets_to_plot = ['k4_q950105', 'mmbody0105']
    
    for dataset in datasets_to_plot:
        df_ds = df[df['dataset'] == dataset].sort_values('data_pct')
        if len(df_ds) < 2:
            continue
        
        x = df_ds['data_pct'].values * TOTAL_DATA_POINTS
        y = df_ds['test_mpjpe'].values
        
        # Get display label
        display_label = label_map.get(dataset, dataset)
        
        # Plot data points (only points get label for legend)
        line, = ax.plot(x, y, linestyle='', label=display_label)
        color = line.get_color()
        
        # Fit and plot power law (no legend entry)
        a, b, r = fit_power_law(x, y)
        if a is not None:
            x_fit = np.linspace(x.min(), 50000, 100)
            y_fit = a * np.power(x_fit, b)
            ax.plot(x_fit, y_fit, linestyle='--', color=color, marker='None')
            # Print fitting result
            print(f"{display_label}: $y={a:.2f}x^{{{b:.4f}}}$, $r={r:.4f}$")
    
    ax.set_xscale('log')
    ax.set_xlabel('Data quantity')
    ax.set_ylabel('MPJPE (mm)')
    
    # Set xlim to 50000
    ax.set_xlim(3000, 50000)
    
    # Grid styling
    ax.grid(True, which='both', linestyle='--', alpha=0.3)
    ax.grid(True, which='major', linestyle='--', alpha=0.7)
    
    # X-axis tick formatting
    ax.xaxis.set_major_locator(mticker.LogLocator(base=10, numticks=20))
    ax.xaxis.set_minor_locator(mticker.LogLocator(base=10, subs=np.arange(2, 10), numticks=100))
    
    def _log_plain(x, pos):
        try:
            return f"{int(x):,}"
        except Exception:
            return f"{x:g}"
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_log_plain))
    
    def _log_minor_label(x, pos):
        if x <= 0:
            return ""
        exp = np.floor(np.log10(x))
        mant = x / (10**exp)
        if np.isclose(mant, 2.0) or np.isclose(mant, 5.0):
            return _log_plain(x, pos)
        return ""
    ax.xaxis.set_minor_formatter(mticker.FuncFormatter(_log_minor_label))
    
    # Tick params
    ax.tick_params(axis='x', which='major', length=6)
    ax.tick_params(axis='x', which='minor', length=3)
    
    # Legend above the chart (fontsize=18 to match paper figure style)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False, fontsize=18)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.85)
    plt.savefig('data_scaling_simulation.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    print("Saved: data_scaling_simulation.pdf")
else:
    print("No data scaling experiments found (all experiments use 100% data).")
    print("Run train_simulation_batch_different_size.py to generate data scaling experiments.")
